In [ ]:
!pip install torch torchvision torchaudio --upgrade --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install -q transformers datasets evaluate accelerate peft trl bitsandbytes scikit-learn rouge_score bert_score sacrebleu

In [ ]:
import os
# Force all Hugging Face downloads to the persistent workspace
os.environ["HF_HOME"] = "/workspace/hf_cache"

import torch
import re
import json
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ... (Keep your COLORS dictionary and clean_colors function here) ...

# Expanded Color Parser
COLORS = {
    "black": (0, 0, 0), "white": (255, 255, 255), "red": (255, 0, 0), "lime": (0, 255, 0), 
    "blue": (0, 0, 255), "yellow": (255, 255, 0), "cyan": (0, 255, 255), "magenta": (255, 0, 255), 
    "silver": (192, 192, 192), "gray": (128, 128, 128), "maroon": (128, 0, 0), "olive": (128, 128, 0),
    "green": (0, 128, 0), "purple": (128, 0, 128), "teal": (0, 128, 128), "navy": (0, 0, 128), 
    "darkblue": (0, 0, 139), "lightblue": (173, 216, 230), "orange": (255, 165, 0), 
    "pink": (255, 192, 203), "brown": (165, 42, 42),
}

def clean_colors(text):
    hex_pattern = re.compile(r'#[0-9a-fA-F]{3,6}\b')
    def get_closest(hex_code):
        try:
            h = hex_code.lstrip('#')
            if len(h) == 3: h = ''.join([c*2 for c in h])
            rgb = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
            return min(COLORS.keys(), key=lambda n: sum((c1-c2)**2 for c1,c2 in zip(rgb, COLORS[n])))
        except: return hex_code
    return hex_pattern.sub(lambda m: get_closest(m.group(0)), text)

In [ ]:
def load_qwen_data(file_path, is_training=True):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # --- UPDATED DICTIONARY KEYS ---
    # We replaced 'text' and 'target_expl' with the exact names SFTTrainer expects
    formatted = {"prompt": [], "completion": [], "target_label": []}
    
    for item in data:
        clean_md = clean_colors(item.get("enriched_markdown", ""))
        lines = clean_md.split('\n')
        compact_md = " ; ".join([l.replace('|', ' ').strip() for l in lines if '| :---' not in l and l.strip()])
        
        claim = item.get("claim", "")
        title = " ".join(str(item.get("title_description", "")).split()[:50])
        label = "SUPPORTED" if item.get("label") == True else "REFUTED"
        explanation = item.get("explanation", "")
        
        # 1. The Prompt (What the model reads)
        sys_msg = "You are an expert data analyst. Verify the claim based on the provided chart data. Explain your reasoning step-by-step, then conclude exactly with 'Label: SUPPORTED' or 'Label: REFUTED'."
        user_msg = f"Title: {title}\nContext: {compact_md}\nClaim: {claim}"
        
        prompt = f"<|im_start|>system\n{sys_msg}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n"
        
        # 2. The Completion (What the model learns to write)
        completion = f"Explanation: {explanation} Label: {label}.<|im_end|>"
        
        formatted["prompt"].append(prompt)
        formatted["completion"].append(completion) # <--- SFTTrainer looks for this specific key
        formatted["target_label"].append(label)
        
    return Dataset.from_dict(formatted)

dataset = DatasetDict({
    "train": load_qwen_data("train_w_spec.json"),
    "validation": load_qwen_data("validation_w_spec.json"),
    "test_1": load_qwen_data("test_1_w_spec.json", is_training=False),
    "test_2": load_qwen_data("test_2_w_spec.json", is_training=False)
})

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --- PyTorch < 2.4 Compatibility Patch ---
# Temporarily patches the missing function required by the newest 'transformers' version
if not hasattr(torch.nn.Module, "set_submodule"):
    def set_submodule(self, target, module):
        atoms = target.split(".")
        name = atoms.pop(-1)
        mod = self
        for item in atoms:
            mod = getattr(mod, item)
        setattr(mod, name, module)
    torch.nn.Module.set_submodule = set_submodule
# -----------------------------------------

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 1. Tokenizer Setup
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 2. 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # RTX 3090 fully supports bf16
)

# 3. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Automatically utilizes the 24GB RTX 3090
)
model = prepare_model_for_kbit_training(model)

# 4. LoRA Adapter Config (Targets the attention and MLP layers for maximum reasoning)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

# 1. We replace TrainingArguments with SFTConfig
training_args = SFTConfig(
    output_dir="/workspace/qwen_results",
    
    # --- STEP-BASED EVALUATION ---
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=20,
    load_best_model_at_end=True, 
    # -----------------------------
    
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    bf16=True, 
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    num_train_epochs=3,
    
    # --- THE FIX: Updated parameter name for new TRL versions ---
    max_length=4096, 
    dataset_text_field="text",
)

# 2. SFTTrainer takes the config arguments
# 2. SFTTrainer now only takes the models, datasets, and the config arguments
# We removed peft_config since the model was already wrapped in Cell 4
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    args=training_args,
)

# Start training!
trainer.train()

In [ ]:
import evaluate
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

# Load all required NLG metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")
bert_metric = evaluate.load("bertscore")

def evaluate_qwen(split_name):
    print(f"\nEvaluating {split_name} using Qwen2.5-7B...")
    
    model.eval()
    test_data = dataset[split_name]
    preds = []
    
    # 1. Generation Loop
    for prompt in tqdm(test_data["prompt"]):
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=256, 
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated response
        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        preds.append(generated_text)
        
    # 2. Parse Predictions
    p_labels, p_expls = [], []
    for p in preds:
        lbl = "SUPPORTED" if "Label: SUPPORTED" in p else "REFUTED"
        expl = p.split("Label:")[0].replace("Explanation:", "").strip()
        p_labels.append(lbl)
        p_expls.append(expl)
        
    t_labels = test_data["target_label"]
    t_expls = test_data["target_expl"]
    
    # 3. Calculate Classification Metrics
    acc = accuracy_score(t_labels, p_labels) * 100
    f1 = f1_score(t_labels, p_labels, average='macro') * 100
    
    # 4. Calculate NLG Metrics
    print("Calculating NLG Metrics (BLEU, ROUGE, BERTScore)...")
    rouge_l = rouge_metric.compute(predictions=p_expls, references=t_expls)['rougeL'] * 100
    bleu = bleu_metric.compute(predictions=p_expls, references=[[t] for t in t_expls])['score']
    
    bert_score = bert_metric.compute(predictions=p_expls, references=t_expls, lang="en")
    avg_bert = np.mean(bert_score['f1']) * 100
    
    # 5. Save to CSV
    results_df = pd.DataFrame({
        "prompt": test_data["prompt"],
        "true_label": t_labels, "pred_label": p_labels,
        "true_explanation": t_expls, "pred_explanation": p_expls,
        "raw_generation": preds
    })
    results_df.to_csv(f"qwen_results_{split_name}.csv", index=False)
    
    return {"acc": acc, "f1": f1, "bleu": bleu, "rouge": rouge_l, "bert": avg_bert, "report": classification_report(t_labels, p_labels, digits=4)}

# Run Evaluations
res1 = evaluate_qwen("test_1")
res2 = evaluate_qwen("test_2")

# --- FINAL QWEN2.5-7B MATRIX ---
avg_acc = (res1['acc'] + res2['acc']) / 2
avg_bleu = (res1['bleu'] + res2['bleu']) / 2
avg_rouge = (res1['rouge'] + res2['rouge']) / 2
avg_bert = (res1['bert'] + res2['bert']) / 2

print("\n" + "="*115)
print("  CHARTCHECK EVALUATION MATRIX — Qwen2.5-7B Results")
print("="*115)
print(f"{'Model':>28} {'Task':>5} {'Test1_Acc':>9} {'Test1_F1':>9} {'Test2_Acc':>9} {'Test2_F1':>9} {'Avg_Acc':>8} {'BLEU':>6} {'ROUGE-L':>7} {'BERTScore':>9}")
print(f"{'Qwen2.5-7B (QLoRA)':>28} {'M':>5} {res1['acc']:>9.1f} {res1['f1']:>9.1f} {res2['acc']:>9.1f} {res2['f1']:>9.1f} {avg_acc:>8.1f} {avg_bleu:>6.1f} {avg_rouge:>7.1f} {avg_bert:>9.1f}")
print("="*115)

print(f"\nTest 1 — Per-class report:\n{res1['report']}")

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"

import torch
import json
import re
import pandas as pd
import numpy as np
import evaluate
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 1. Re-declare Color Parser and Data Loader (since RAM was cleared)
COLORS = {"black": (0, 0, 0), "white": (255, 255, 255), "red": (255, 0, 0), "lime": (0, 255, 0), "blue": (0, 0, 255), "yellow": (255, 255, 0), "cyan": (0, 255, 255), "magenta": (255, 0, 255), "silver": (192, 192, 192), "gray": (128, 128, 128), "maroon": (128, 0, 0), "olive": (128, 128, 0), "green": (0, 128, 0), "purple": (128, 0, 128), "teal": (0, 128, 128), "navy": (0, 0, 128), "darkblue": (0, 0, 139), "lightblue": (173, 216, 230), "orange": (255, 165, 0), "pink": (255, 192, 203), "brown": (165, 42, 42)}

def clean_colors(text):
    hex_pattern = re.compile(r'#[0-9a-fA-F]{3,6}\b')
    def get_closest(hex_code):
        try:
            h = hex_code.lstrip('#')
            if len(h) == 3: h = ''.join([c*2 for c in h])
            rgb = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
            return min(COLORS.keys(), key=lambda n: sum((c1-c2)**2 for c1,c2 in zip(rgb, COLORS[n])))
        except: return hex_code
    return hex_pattern.sub(lambda m: get_closest(m.group(0)), text)

def load_qwen_test_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    formatted = {"prompt": [], "target_label": [], "target_expl": []}
    for item in data:
        clean_md = clean_colors(item.get("enriched_markdown", ""))
        lines = clean_md.split('\n')
        compact_md = " ; ".join([l.replace('|', ' ').strip() for l in lines if '| :---' not in l and l.strip()])
        claim, title, label, explanation = item.get("claim", ""), " ".join(str(item.get("title_description", "")).split()[:50]), "SUPPORTED" if item.get("label") == True else "REFUTED", item.get("explanation", "")
        sys_msg = "You are an expert data analyst. Verify the claim based on the provided chart data. Explain your reasoning step-by-step, then conclude exactly with 'Label: SUPPORTED' or 'Label: REFUTED'."
        user_msg = f"Title: {title}\nContext: {compact_md}\nClaim: {claim}"
        formatted["prompt"].append(f"<|im_start|>system\n{sys_msg}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n")
        formatted["target_label"].append(label)
        formatted["target_expl"].append(explanation)
    return Dataset.from_dict(formatted)

dataset = DatasetDict({
    "test_1": load_qwen_test_data("test_1_w_spec.json"),
    "test_2": load_qwen_test_data("test_2_w_spec.json")
})

# 2. Reload Model & LoRA Weights safely
print("Loading model and LoRA weights...")
model_id = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/workspace/qwen_results" # Path where your TRL trainer saved the final weights

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir="/workspace/hf_cache/models")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # <--- CRITICAL: Required for batched generation

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto", 
    cache_dir="/workspace/hf_cache/models",
    attn_implementation="sdpa" # <--- PyTorch Native Flash Attention for massive speedup
)

# Merge the trained adapter into the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# 3. Load Metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")
bert_metric = evaluate.load("bertscore")

# 4. Optimized Evaluation Loop
def evaluate_qwen_fast(split_name, batch_size=8): # Processing 8 at a time!
    print(f"\nEvaluating {split_name} using Qwen2.5-7B (Batched)...")
    test_data = dataset[split_name]
    preds = []
    
    # Batch processing loop
    for i in tqdm(range(0, len(test_data["prompt"]), batch_size)):
        batch_prompts = test_data["prompt"][i:i+batch_size]
        
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=4096).to("cuda")
        
        with torch.inference_mode(): # Faster than no_grad
            outputs = model.generate(
                **inputs, 
                max_new_tokens=256, 
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )
            
        # Extract only the newly generated text for the whole batch
        input_lengths = inputs.input_ids.shape[1]
        generated_tokens = outputs[:, input_lengths:]
        batch_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        preds.extend(batch_preds)
        
    # Parse Predictions
    p_labels, p_expls = [], []
    for p in preds:
        lbl = "SUPPORTED" if "Label: SUPPORTED" in p else "REFUTED"
        expl = p.split("Label:")[0].replace("Explanation:", "").strip()
        p_labels.append(lbl)
        p_expls.append(expl)
        
    t_labels = test_data["target_label"]
    t_expls = test_data["target_expl"]
    
    acc = accuracy_score(t_labels, p_labels) * 100
    f1 = f1_score(t_labels, p_labels, average='macro') * 100
    
    print("Calculating NLG Metrics (BLEU, ROUGE, BERTScore)...")
    rouge_l = rouge_metric.compute(predictions=p_expls, references=t_expls)['rougeL'] * 100
    bleu = bleu_metric.compute(predictions=p_expls, references=[[t] for t in t_expls])['score']
    bert_score = bert_metric.compute(predictions=p_expls, references=t_expls, lang="en")
    avg_bert = np.mean(bert_score['f1']) * 100
    
    results_df = pd.DataFrame({
        "prompt": test_data["prompt"],
        "true_label": t_labels, "pred_label": p_labels,
        "true_explanation": t_expls, "pred_explanation": p_expls,
        "raw_generation": preds
    })
    results_df.to_csv(f"qwen_results_{split_name}.csv", index=False)
    
    return {"acc": acc, "f1": f1, "bleu": bleu, "rouge": rouge_l, "bert": avg_bert, "report": classification_report(t_labels, p_labels, digits=4)}

# 5. Run it
res1 = evaluate_qwen_fast("test_1", batch_size=8)
res2 = evaluate_qwen_fast("test_2", batch_size=8)

# --- FINAL QWEN2.5-7B MATRIX ---
avg_acc = (res1['acc'] + res2['acc']) / 2
avg_bleu = (res1['bleu'] + res2['bleu']) / 2
avg_rouge = (res1['rouge'] + res2['rouge']) / 2
avg_bert = (res1['bert'] + res2['bert']) / 2

print("\n" + "="*115)
print("  CHARTCHECK EVALUATION MATRIX — Qwen2.5-7B Results")
print("="*115)
print(f"{'Model':>28} {'Task':>5} {'Test1_Acc':>9} {'Test1_F1':>9} {'Test2_Acc':>9} {'Test2_F1':>9} {'Avg_Acc':>8} {'BLEU':>6} {'ROUGE-L':>7} {'BERTScore':>9}")
print(f"{'Qwen2.5-7B (QLoRA)':>28} {'M':>5} {res1['acc']:>9.1f} {res1['f1']:>9.1f} {res2['acc']:>9.1f} {res2['f1']:>9.1f} {avg_acc:>8.1f} {avg_bleu:>6.1f} {avg_rouge:>7.1f} {avg_bert:>9.1f}")
print("="*115)

print(f"\nTest 1 — Per-class report:\n{res1['report']}")
print(f"\nTest 2 — Per-class report:\n{res2['report']}")